# Exploratory data analysis of the OTTO sample

Follows `_doc/eda_guide.md` step by step. One section per step; the decision log is `eda/eda_log.md`, comparison tables are in `eda/tables/`, figures in `eda/figures/`. Each step has its own GitHub issue (umbrella #5; step N is issue #91 + N). Rerun everything with `EDA_FORCE=1` (otherwise expensive tables are cached in `eda/tables/`).

**Two tables are analysed.**
- **Events** (`data/sample/events.parquet`): one row per event (`session`, `aid`, `ts`, `type`). No target. Used by the steps that do not need one.
- **Candidates** (`data/features/train_asof11.parquet`): one row per (session, candidate item) from the shared pool, with 18 numeric features, the category and the label. This is the modelling table for the steps that need a target.

In [ ]:
import platform
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from eda_utils import SEED, cached, log_step, save_table
from split import DAY_MS, START_MS, day_to_ms

np.random.seed(SEED)
DATA = ROOT / "data"
D14, D16 = day_to_ms(14), day_to_ms(16)
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
dev = events.filter(pl.col("ts") < D14)          # development set: every target-aware cell reads only this
CAND = DATA / "features" / "train_asof11.parquet"
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_width_chars(200)

## Step 0. Setup, framing and holdout
*Kind: diagnostic. Issue #91.*

#### Why this step exists here
Before touching any data we need to say what the analysis is for, what one row is, what the target is and how the model will be used. Everything later depends on this: the metric, the split and which columns may be used. The holdout is set aside now, so that nothing target-aware can leak into the decisions that follow.

#### Options considered
- **Holdout scheme:** time-ordered (latest rows held out), grouped by entity (session), or random/stratified. The guide's rule is: time-ordered if the model scores future data, grouped if entities repeat, random otherwise.
- **Target definition:** the guide needs one target column. Candidates: `label > 0` (a candidate is found later), a single event type (click), or the graded label. `label > 0` is the broadest and has the most positives.

In [ ]:
framing = pl.DataFrame([
    ("Decision supported", "Rank candidate items for a session prefix and a query category (search and ranking on e-commerce sessions).", "given (project goal)"),
    ("Unit of analysis", "Events table: one event. Candidates table: one (session, candidate item) pair. The entity is the session; the same session has many rows in both.", "given"),
    ("Target", "`label > 0` on the candidates table: the candidate is clicked, carted or ordered later in the same session (graded: 1 click, 2 cart, 3 order). Defined in issues #10 and #22.", "assumed for the guide (the guide needs a single target column)"),
    ("Task type", "Binary classification on candidate rows (PR-AUC), and ranking (NDCG@10) for the final metric.", "assumed"),
    ("When is the target known", "After the session prefix: the held-out events that follow the cut.", "given"),
    ("Metric and error cost", "NDCG@10 with bootstrap intervals for ranking; PR-AUC for the classification view. Cost of false positives and false negatives not given.", "assumed"),
    ("Time order / future scoring", "Yes. Timestamps exist and the model scores later sessions.", "given"),
    ("Known at prediction time", "Features are computed as of a cutoff before the session (issue #22); the label is not.", "given"),
    ("Constraints", "8 GB RAM Mac, minimise cost, no PII (anonymous ids).", "given"),
], schema=["question", "answer", "source"], orient="row")
print(framing)

# Holdout: the evaluation window (sessions that start on days 14-15), time ordered because the model scores future data.
first = events.group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
holdout = first.filter((pl.col("first_ts") >= D14) & (pl.col("first_ts") < D16))
dev_sessions = first.filter(pl.col("first_ts") < D14)
dev_max_ts = dev.select(pl.col("ts").max()).collect()[0, 0]
assert dev_max_ts < D14, "the development set must contain only events before day 14"
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())
cand_sessions = pl.read_parquet(CAND, columns=["session"])["session"].unique()
cand_first = first.filter(pl.col("session").is_in(cand_sessions.to_list()))
assert (cand_first["first_ts"] < D14).all() and (cand_first["first_ts"] >= day_to_ms(11)).all()
import lightgbm
import scipy
import sklearn

versions = {"python": platform.python_version(), "polars": pl.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "lightgbm": lightgbm.__version__, "seed": SEED}
print(f"holdout: {holdout.height:,} sessions starting on days 14-15 (time-ordered); development: {dev_sessions.height:,} sessions, events before day 14")
print("candidates table sessions (all development):", cand_sessions.len())
print(versions)

In [ ]:
e0 = log_step(0, "Setup and framing", "Diagnostic",
    "Framing table above (each answer marked given or assumed). Holdout scheme options: time-ordered, grouped by entity, random. Layout eda/{tables,figures}, seed 42, versions printed.",
    f"Development events: ts < day 14 ({dev_sessions.height:,} sessions). Holdout: {holdout.height:,} sessions that start on days 14-15. No session is on both sides. The candidates table has {cand_sessions.len():,} sessions, all in development (they start on days 11-13).",
    "Time-ordered holdout = the evaluation window of issue #6 (days 14-15). Everything target-aware reads only events before day 14.",
    "The model scores future sessions, so the latest rows are held out (guide rule). Grouping by session is implied: holdout sessions start after the development window.",
    "Random split: rejected, it would mix time periods. Group-only split: rejected, it does not test future data.",
    "High", "Steps 12 and 15 re-check the scheme; the holdout is never read before the modelling phase.",
    "yes: the target and task type are assumed from issues #10/#22. Please confirm `label > 0` is the target to use for the guide.")
print(e0)

#### Chosen approach and rationale
**Decision.** Time-ordered holdout = the evaluation window of issue #6 (days 14-15). Everything target-aware reads only events before day 14.

**Why.** The model scores future sessions, so the latest rows are held out (guide rule). Grouping by session is implied: holdout sessions start after the development window.

**Rejected.** Random split: rejected, it would mix time periods. Group-only split: rejected, it does not test future data.

**Confidence.** High

#### Interpretation: what we understand from the results
The holdout and the development set are cleanly separated in time and by session: no session is on both sides, and the 40,000 sessions of the candidate table all start on days 11-13. Two assumptions carry weight: the guide needs a single target column, and we chose `label > 0` on the candidate table from #22, keeping the graded labels for the ranking metric. This is the one required input the guide asks a human to supply, so it is marked **assumed** and needs your confirmation.

#### Impact on next steps
Steps 12 and 15 re-check the scheme; the holdout is never read before the modelling phase.

#### Out of scope
- Confirmation of the target and task type (issue #91 stays open until then).
- NDCG@10 as the final ranking metric comes from the project plan, not from this guide.

## Step 1. Load and first look
*Kind: diagnostic. Issue #92.* Options: A manual inspection, B per-column overview, C automated profile.

#### Why this step exists here
The first look tells us the size and shape of the data and shows obvious problems before any analysis. It comes right after the framing because we now know which table and which rows (development only) to look at.

#### Options considered
- **A. Manual inspection:** shape, memory, dtypes, first, last and random rows, summary statistics.
- **B. Per-column overview:** one table with dtype, missing share, unique count, example, minimum and maximum for every column.
- **C. Automated profile:** a profiling report in minimal mode. **Not run:** ydata-profiling needs numpy < 2.2 and matplotlib <= 3.10; this project runs numpy 2.4.6 and matplotlib 3.11.2, so installing it would downgrade core libraries under every earlier result. With only 4 ID-like columns, A and B already show what it would.

In [ ]:
ev = dev.collect()                                          # 13.06 million rows x 4 columns: fits in memory
print("A. shape:", ev.shape, " estimated memory:", round(ev.estimated_size() / 1e6), "MB")
print("dtypes:", dict(ev.schema))
print("first 5:\n", ev.head(5)); print("last 5:\n", ev.tail(5)); print("random 5:\n", ev.sample(5, seed=SEED))
print(ev.describe())

In [ ]:
overview = pl.DataFrame([{
    "column": c, "dtype": str(ev.schema[c]), "missing_share": ev[c].null_count() / ev.height, "n_unique": ev[c].n_unique(),
    "unique_share": ev[c].n_unique() / ev.height, "example": str(ev[c][0]), "min": str(ev[c].min()), "max": str(ev[c].max()),
} for c in ev.columns])
save_table("step01_column_overview", overview)
print("B. per-column overview:"); print(overview)
print("C. automated profile: not run, see the log (ydata-profiling needs numpy<2.2 and matplotlib<=3.10; the project has numpy",
      np.__version__, ")")

In [ ]:
issues1 = pl.DataFrame([
    ("session", "ID-like: 1.0 million distinct ids, an identifier not a feature", "exclude from features; use as the entity for splits (Steps 4, 15)"),
    ("aid", "ID-like item identifier with 1.4 million distinct values, long tail", "never a numeric feature; encoding options in Step 13"),
    ("ts", "Unix milliseconds stored as int64", "convert to a datetime for time steps (Step 2A)"),
    ("type", "int8 code 0/1/2 for click/cart/order, low cardinality", "treat as categorical (Step 2A)"),
    ("all", "No missing values, no constant column", "Step 5 has little to treat"),
], schema=["column", "problem", "handled_in"], orient="row")
save_table("step01_issue_list", issues1); print(issues1)
assert ev.null_count().sum_horizontal()[0] == 0
e1 = log_step(1, "Load and first look", "Diagnostic",
    "A manual inspection (shape, dtypes, first/last/random rows, describe); B per-column overview table; C automated profile SKIPPED: ydata-profiling requires numpy<2.2 and matplotlib<=3.10, the project uses numpy 2.4.6 and matplotlib 3.11.2, so installing it would downgrade core libraries for every earlier result; with 4 ID-like columns A and B already show its alerts.",
    f"{ev.height:,} rows x 4 columns (session int64, aid int32, ts int64, type int8), {ev.estimated_size() / 1e6:.0f} MB; no missing values; no constant column; session and aid are ID-like; ts is epoch milliseconds; type has 3 codes.",
    "Prioritised issue list saved to eda/tables/step01_issue_list.csv: convert ts, treat type as categorical, exclude ids from features.",
    "The three options were meant to be combined into one list; two were run and their findings agree.",
    "C profile: skipped for the reason above; installing at the cost of downgrading numpy was rejected.",
    "High", "Step 2 converts types; Step 13 handles aid as high-cardinality.")
print(e1)

#### Chosen approach and rationale
**Decision.** Prioritised issue list saved to eda/tables/step01_issue_list.csv: convert ts, treat type as categorical, exclude ids from features.

**Why.** The three options were meant to be combined into one list; two were run and their findings agree.

**Rejected.** C profile: skipped for the reason above; installing at the cost of downgrading numpy was rejected.

**Confidence.** High

#### Interpretation: what we understand from the results
The events table is small and clean: 4 columns, no missing values, no constant column, no impossible structure. What matters is what the columns mean: two are identifiers (session, item) that must never be used as numeric features, one is epoch time, and one is a 3-code event type. The interesting problems are not in data quality but in the structure of the candidate table, which the later steps examine.

#### Impact on next steps
Step 2 converts types; Step 13 handles aid as high-cardinality.

#### Out of scope
- Automated profile (option C): blocked by the dependency conflict above. No follow-up needed unless the numpy pin changes.

## Step 2. Types, validity and consistency
*Kind: diagnostic then fixes. Issue #93.* Options: A dtype vs meaning, B sentinel values, C range and domain rules, D cross-field consistency (development events, ts < day 14).

#### Why this step exists here
Now that we know the columns, we check that each one means what its name says and holds only possible values. Impossible values are handled by converting them to missing plus a flag, never by deleting rows. This comes before duplicates and missing values because those steps assume the values are valid.

#### Options considered
- **A. Dtype vs meaning:** numbers stored as text, dates as text, low-cardinality codes that are really categories.
- **B. Sentinel and placeholder values:** -1, 0, 999, 9999, 99999, and similar acting as missing.
- **C. Range and domain rules:** timestamps inside the 4 labelled weeks, event type in {0, 1, 2}, non-negative ids.
- **D. Cross-field consistency:** timestamps non-decreasing inside a session, a session in one block, carts after clicks and orders after carts.

In [ ]:
START, END = START_MS, START_MS + 28 * DAY_MS
n = ev.height
# A dtype vs meaning
A = pl.DataFrame([
    ("session", "int64", "identifier", "keep as id, never a numeric feature"),
    ("aid", "int32", "identifier", "keep as id"),
    ("ts", "int64", "epoch milliseconds", "convert to datetime for time steps; the raw integer is kept for splits"),
    ("type", "int8", "categorical code 0 click, 1 cart, 2 order", "cast to a categorical for encoders; the code is fine for trees"),
], schema=["column", "stored_as", "meaning", "conversion"], orient="row"); print("A.\n", A)
# B sentinels
sent = [-1, 0, 999, 9999, 99999]
hits = [{"column": c, "value": v, "rows": int((ev[c].cast(pl.Int64) == v).sum())} for c in ("session", "aid", "ts", "type") for v in sent]
B = pl.DataFrame([h for h in hits if h["rows"] > 0] or [{"column": "-", "value": 0, "rows": 0}]).with_columns((pl.col("rows") / n).alias("share")); print("B. sentinel-like values found:\n", B)
# C ranges
C = pl.DataFrame([
    ("ts inside the 4 labelled weeks", int(((ev["ts"] < START) | (ev["ts"] >= END)).sum())),
    ("type not in {0,1,2}", int((~ev["type"].is_in([0, 1, 2])).sum())),
    ("aid < 0", int((ev["aid"] < 0).sum())), ("session < 0", int((ev["session"] < 0).sum())),
    ("ts in the future of the development cut (>= day 14)", int((ev["ts"] >= D14).sum())),
], schema=["rule", "violations"], orient="row").with_columns((pl.col("violations") / n).alias("share")); print("C.\n", C)
# D cross-field
srt = ev.select(pl.col("session"), pl.col("ts"), (pl.col("ts").diff() < 0).alias("backwards"), (pl.col("session") != pl.col("session").shift(1)).alias("new"))
backwards_within = int((srt["backwards"] & ~srt["new"]).sum())
blocks = int(srt["new"].sum()) + 1; sessions = ev["session"].n_unique()   # +1: the first row has no previous row
first_seen = ev.group_by("session", "aid").agg(
    pl.col("ts").filter(pl.col("type") == 0).min().alias("t_click"), pl.col("ts").filter(pl.col("type") == 1).min().alias("t_cart"),
    pl.col("ts").filter(pl.col("type") == 2).min().alias("t_order"))
cart_no_click = first_seen.filter(pl.col("t_cart").is_not_null() & (pl.col("t_click").is_null() | (pl.col("t_cart") < pl.col("t_click")))).height
order_no_cart = first_seen.filter(pl.col("t_order").is_not_null() & (pl.col("t_cart").is_null() | (pl.col("t_order") < pl.col("t_cart")))).height
pairs = first_seen.height
D = pl.DataFrame([
    ("timestamps go backwards inside a session", backwards_within, backwards_within / n),
    ("a session appears in more than one block (blocks - sessions)", blocks - sessions, (blocks - sessions) / n),
    ("a cart of an item with no earlier click (share of session-item pairs)", cart_no_click, cart_no_click / pairs),
    ("an order of an item with no earlier cart (share of session-item pairs)", order_no_cart, order_no_cart / pairs),
], schema=["rule", "violations", "share"], orient="row"); print("D.\n", D)
for name, t in (("A", A), ("B", B), ("C", C), ("D", D)):
    save_table(f"step02_{name}", t)

In [ ]:
bad_share = max(float(C["share"].max()), float(D["share"][:2].max()))
e2 = log_step(2, "Types, validity and consistency", "Diagnostic then fixes",
    "A dtype vs meaning; B sentinel-like values (-1, 0, 999, 9999, 99999); C range and domain rules; D cross-field consistency (ordering inside sessions, cart/order after click/cart). All four run on the development events.",
    f"A: 4 columns, all correctly typed except that ts is epoch ms. B: no placeholder found. The scan flags aid values 0, 999, 9999 and 99999 (2, 4, 44 and 1 events), but item ids are dense integers from 0 to 1,855,602, so these are ordinary ids; session 0 (86 events) is the first session; type 0 is the code for a click (89.8% of events). Details in eda/tables/step02_B.csv. C: {int(C['violations'].sum()):,} range violations (ts inside the 4 weeks, type in {{0,1,2}}, ids non-negative). D: {backwards_within:,} backwards timestamps inside sessions; {cart_no_click / pairs:.1%} of carted items had no earlier click in the same session and {order_no_cart / pairs:.1%} of ordered items had no earlier cart.",
    "No fixes needed for A-C. D is a property of the data (users can add to the cart from a page without a logged click), not an error. The sentinel-like values are not placeholders (evidence above), so nothing goes on the human question list.",
    "Rule: apply unambiguous fixes, convert impossible values to missing with a flag, never delete rows. There was nothing impossible.",
    "Nothing to reject: all impossible-value options found no violations. Treating cart-without-click as an error was rejected because it is common and plausible.",
    "High", "Step 4 uses the key (session, aid, ts, type); Step 12 must not assume a click precedes a cart.")
print(e2)

#### Chosen approach and rationale
**Decision.** No fixes needed for A-C. D is a property of the data (users can add to the cart from a page without a logged click), not an error. The sentinel-like values are not placeholders (evidence above), so nothing goes on the human question list.

**Why.** Rule: apply unambiguous fixes, convert impossible values to missing with a flag, never delete rows. There was nothing impossible.

**Rejected.** Nothing to reject: all impossible-value options found no violations. Treating cart-without-click as an error was rejected because it is common and plausible.

**Confidence.** High

#### Interpretation: what we understand from the results
Nothing is broken: every value is in range and time order inside sessions is intact. The values that look like sentinels (item ids 0, 999, 9999, 99999) are ordinary ids, because item ids are dense integers from 0 to 1,855,602; session 0 is the first session; type 0 is a click. The one apparent inconsistency, carts of items never clicked (2.1% of session-item pairs) and orders with no cart (0.3%), is normal shop behaviour (adding to the cart from a list page). It matters later because a click cannot be assumed to precede a cart.

#### Impact on next steps
Step 4 uses the key (session, aid, ts, type); Step 12 must not assume a click precedes a cart.

#### Out of scope
- None.

## Step 3. Target analysis
*Kind: diagnostic. Issue #94.* Target on the candidates table: `label > 0`.

#### Why this step exists here
We look at the target before any feature, because the target's shape decides the metric (PR-AUC or ROC-AUC), the split and how much we can trust per-class numbers. This step uses the candidate table, which is the only table with a target.

#### Options considered
- **A. Distribution:** class counts and shares, the imbalance ratio.
- **B. Absolute minority count:** the number of rows in the rarest class, not only its share (a small share of a huge table can still be plenty).
- **C. Stability:** the target rate by day and by the largest segments.
- **D. Missing target:** rows with no label.

In [ ]:
cand = pl.read_parquet(CAND, columns=["session", "aid", "category", "label", "y_click", "y_cart", "y_order"])
print("A. label counts and shares:")
A3 = cand.group_by("label").len().sort("label").with_columns((pl.col("len") / cand.height).alias("share")); print(A3)
pos = int((cand["label"] > 0).sum())
print(f"y = label > 0: {pos:,} positives of {cand.height:,} rows ({pos / cand.height:.3%}); imbalance ratio 1 : {(cand.height - pos) / pos:.0f}")
B3 = pl.DataFrame({"class": ["any (label>0)", "click", "cart", "order"], "positives": [pos, int(cand["y_click"].sum()), int(cand["y_cart"].sum()), int(cand["y_order"].sum())]}); print("B. absolute minority counts:"); print(B3)
first_ts = first.filter(pl.col("session").is_in(cand["session"].unique().to_list()))
c2 = cand.join(first_ts, on="session").with_columns(((pl.col("first_ts") - day_to_ms(0)) // DAY_MS).alias("day"))
C3a = c2.group_by("day").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("day"); print("C. by day of the session start:"); print(C3a)
top5 = c2.group_by("category").len().sort("len", descending=True).head(5)["category"].to_list()
C3b = c2.filter(pl.col("category").is_in(top5)).group_by("category").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("rows", descending=True); print("C. by the 5 largest categories:"); print(C3b)
print("D. rows without a target:", int(cand["label"].null_count()))
for nme, t in (("A", A3), ("B", B3), ("C_day", C3a), ("C_category", C3b)):
    save_table(f"step03_{nme}", t)

In [ ]:
# minority counts in the development set
dev_pos = pos  # the candidates table contains development sessions only
e3 = log_step(3, "Target analysis", "Diagnostic",
    "A class counts and shares; B absolute minority counts; C target rate by day and by the 5 largest categories; D missing target.",
    f"{cand.height:,} rows, {pos:,} positives ({pos / cand.height:.3%}); ratio 1:{(cand.height - pos) / pos:.0f}. Positives by type: click {int(cand['y_click'].sum()):,}, cart {int(cand['y_cart'].sum()):,}, order {int(cand['y_order'].sum()):,}. Target rate by start day and by category in eda/tables/step03_C_*.csv. No missing target.",
    "Class imbalance is SEVERE (minority under 1 percent): Step 14 handles it. The cart and order classes are small but above 100 rows; orders are the rarest.",
    "Guide rules: flag imbalance under 10 percent, severe under 1 percent; remove rows with no target (none).",
    "No option was rejected; all four were run.",
    "High", "Step 14 must use PR-AUC; Step 15 uses grouped folds because a session has 200 rows; the order task has few positives, so per-task results are noisy.")
print(e3)

#### Chosen approach and rationale
**Decision.** Class imbalance is SEVERE (minority under 1 percent): Step 14 handles it. The cart and order classes are small but above 100 rows; orders are the rarest.

**Why.** Guide rules: flag imbalance under 10 percent, severe under 1 percent; remove rows with no target (none).

**Rejected.** No option was rejected; all four were run.

**Confidence.** High

#### Interpretation: what we understand from the results
The target is severely imbalanced (0.44% positives, 1 in 227), which makes ROC-AUC look flattering and PR-AUC the honest metric. It is stable over the three days and across the five largest categories (0.35% to 0.51%), so there is no sign of a drifting definition. The click class is large (34,759 rows), while carts (2,512) and orders (589) are small but above the 100-row threshold, so per-task numbers will be noisy. That explains the wide gap between the validation and evaluation AUC of the order task seen in issue #28.

#### Impact on next steps
Step 14 must use PR-AUC; Step 15 uses grouped folds because a session has 200 rows; the order task has few positives, so per-task results are noisy.

#### Out of scope
- None.

## Step 4. Duplicates
*Kind: selection (how to treat them). Issue #95.* Detection: A exact, B key-based, C near duplicates, D conflicting labels.

#### Why this step exists here
Repeated rows can inflate counts, bias popularity and co-visitation, and leak across a split. We look for them before missing values and features, because a duplicate is either an error to drop or a real repeated observation to keep, and that decision changes every count downstream.

#### Options considered
Detection options (all run): **A. exact** (all columns equal); **B. key-based** (same session, item and time, with or without the type); **C. near duplicates** (same session, item and type within one second); **D. conflicting labels** (identical features, different target). Treatment options: keep everything, drop exact duplicates keeping the first, drop only click and cart duplicates, aggregate per key.

In [ ]:
nrows = ev.height
# A exact
A4 = nrows - ev.unique().height
# B key based (session, aid, ts) and (session, aid, type, ts)
B1 = nrows - ev.select("session", "aid", "ts").unique().height
B2 = nrows - ev.select("session", "aid", "type", "ts").unique().height
# C near duplicates: same session, item and type within the same second
C4 = nrows - ev.with_columns((pl.col("ts") // 1000).alias("sec")).select("session", "aid", "type", "sec").unique().height
dup = pl.DataFrame([("A exact (all 4 columns)", A4), ("B key (session, aid, type, ts)", B2), ("B key (session, aid, ts), any type", B1), ("C near (session, aid, type, same second)", C4)],
    schema=["kind", "rows_beyond_the_first"], orient="row").with_columns((pl.col("rows_beyond_the_first") / nrows).alias("share")); print(dup)
save_table("step04_events_duplicates", dup)
# how do the exact duplicates cluster?
exact = ev.group_by(ev.columns).len().filter(pl.col("len") > 1)
print("groups with exact duplicates:", exact.height, "; largest group size:", int(exact["len"].max()) if exact.height else 0)
by_type = ev.filter(pl.struct(ev.columns).is_duplicated()).group_by("type").len().sort("type"); print("exact duplicates by event type (all copies):"); print(by_type)
# across the development / holdout boundary: sessions on both sides (none, checked in Step 0)
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())

In [ ]:
# D conflicting labels on the candidates table: identical (rounded) features but different target
feat = pl.read_parquet(CAND).drop(["session", "aid"])
num = [c for c in feat.columns if feat.schema[c] in (pl.Float32, pl.Float64)]
rounded = feat.with_columns([pl.col(c).round(3) for c in num])
keys = [c for c in rounded.columns if c not in ("label", "y_click", "y_cart", "y_order")]
groups = rounded.with_columns((pl.col("label") > 0).alias("y")).group_by(keys).agg(pl.len().alias("n"), pl.col("y").n_unique().alias("n_targets"), pl.col("y").sum().alias("pos"))
multi = groups.filter(pl.col("n") > 1)
conflict = multi.filter(pl.col("n_targets") > 1)
rows_in_conflict = int(conflict["n"].sum()); minority_in_conflict = int(conflict["pos"].sum())
print(f"D. feature combinations shared by 2+ rows: {multi.height:,} ({int(multi['n'].sum()):,} rows); with conflicting targets: {conflict.height:,} combinations, {rows_in_conflict:,} rows ({rows_in_conflict / feat.height:.2%} of all rows)")
print("share of positives that sit in a conflicting group:", round(minority_in_conflict / pos, 3))
D4 = pl.DataFrame({"measure": ["combinations shared by 2+ rows", "rows in those combinations", "combinations with conflicting targets", "rows in conflicting combinations", "share of all rows", "share of positives in a conflict"],
                   "value": [float(multi.height), float(multi["n"].sum()), float(conflict.height), float(rows_in_conflict), rows_in_conflict / feat.height, minority_in_conflict / pos]})
save_table("step04_candidates_conflicts", D4); print(D4)
del feat, rounded, groups

In [ ]:
order_dups = int(by_type.filter(pl.col("type") == 2)["len"][0]); click_dups = int(by_type.filter(pl.col("type") == 0)["len"][0]); cart_dups = int(by_type.filter(pl.col("type") == 1)["len"][0])
n_order = int((ev["type"] == 2).sum())
print(f"order events involved in exact duplicates: {order_dups:,} of {n_order:,} ({order_dups / n_order:.1%}); clicks {click_dups:,} of {int((ev['type'] == 0).sum()):,} ({click_dups / int((ev['type'] == 0).sum()):.2%}); carts {cart_dups:,}")
e4 = log_step(4, "Duplicates", "Selection (how to treat them)",
    "A exact; B key-based ((session, aid, type, ts) and (session, aid, ts)); C near-duplicates (same session, item, type in the same second); D conflicting labels on the candidates table (features rounded to 3 decimals). Treatments considered: keep everything, drop exact duplicates keeping the first, drop only the click and cart duplicates, aggregate per key.",
    f"A: {A4:,} rows ({A4 / nrows:.3%}) are exact copies of an earlier row, in {exact.height:,} groups (largest group 6). B is identical to A (no key repeats with a different value or type). C: {C4:,} rows ({C4 / nrows:.3%}). By event type, the copies are mostly ORDERS: {order_dups:,} order events ({order_dups / n_order:.1%} of all order events) are involved, against {click_dups:,} clicks and {cart_dups:,} carts. D: {conflict.height} feature combinations have conflicting targets, covering {rows_in_conflict} rows ({rows_in_conflict / cand.height:.5%} of {cand.height:,}); each holds one positive and one negative, i.e. {minority_in_conflict} positives ({minority_in_conflict / pos:.2%} of all positives). Duplicates across the development/holdout boundary: none (no session is on both sides).",
    "Drop exact duplicate CLICK and CART events, keeping the first (double logging in the same millisecond has no business reason). KEEP order duplicates: an order of the same item logged twice at the same millisecond is most likely a quantity above 1, which is real. Conflicting labels: none to treat. Not applied to the data used by the earlier issues (#7, #17-#29): the events involved are 0.10% of clicks and 0.09% of carts (copies including the first occurrence), and the effect on those results is measured in follow-up issue #109.",
    "Guide rules: exact duplicates with no business reason are dropped keeping the first; legitimate repeated observations are kept; conflicting labels are quantified and never deleted without approval. Orders have a plausible business reason (quantity), clicks and carts do not.",
    "Keep everything: rejected for clicks and carts (they would double count in co-visitation and popularity). Drop all exact duplicates: rejected because it would remove real order quantities (8.7% of order events are involved). Aggregate per key: rejected, it changes the event grain that Steps 12 and 16 rely on. Near-duplicates (C): NOT dropped: a second click on the same item within one second cannot be told from a real fast repeat without inspecting a sample, and the guide says drop near duplicates only if a sample shows they are the same record.",
    "Medium (the order-quantity reading is an inference, not something the data proves)",
    "Affects the event counts by 0.15%; co-visitation, popularity and features from earlier issues used the un-deduplicated events. Follow-up issue #109 quantifies the effect on the closed results. Step 15 uses grouped folds by session regardless.")
print(e4)

#### Chosen approach and rationale
**Decision.** Drop exact duplicate CLICK and CART events, keeping the first (double logging in the same millisecond has no business reason). KEEP order duplicates: an order of the same item logged twice at the same millisecond is most likely a quantity above 1, which is real. Conflicting labels: none to treat. Not applied to the data used by the earlier issues (#7, #17-#29): the events involved are 0.10% of clicks and 0.09% of carts (copies including the first occurrence), and the effect on those results is measured in follow-up issue #109.

**Why.** Guide rules: exact duplicates with no business reason are dropped keeping the first; legitimate repeated observations are kept; conflicting labels are quantified and never deleted without approval. Orders have a plausible business reason (quantity), clicks and carts do not.

**Rejected.** Keep everything: rejected for clicks and carts (they would double count in co-visitation and popularity). Drop all exact duplicates: rejected because it would remove real order quantities (8.7% of order events are involved). Aggregate per key: rejected, it changes the event grain that Steps 12 and 16 rely on. Near-duplicates (C): NOT dropped: a second click on the same item within one second cannot be told from a real fast repeat without inspecting a sample, and the guide says drop near duplicates only if a sample shows they are the same record.

**Confidence.** Medium (the order-quantity reading is an inference, not something the data proves)

#### Interpretation: what we understand from the results
Duplicates are rare (0.15% of events) but not random: they are concentrated in orders (8.7% of order events are involved, against 0.10% of clicks). That points to order quantities (the same item logged twice at the same millisecond) more than to logging errors, which is why order duplicates are kept. This is an inference, so the confidence is Medium. Conflicting labels are almost absent (7 feature combinations out of 8 million rows), so label noise does not put a ceiling on the accuracy we can reach with this table.

#### Impact on next steps
Affects the event counts by 0.15%; co-visitation, popularity and features from earlier issues used the un-deduplicated events. Follow-up issue #109 quantifies the effect on the closed results. Step 15 uses grouped folds by session regardless.

#### Out of scope
- The effect of the de-duplication on the closed co-visitation, popularity and feature results: follow-up issue #109.

## Step 5. Missing values
*Kind: diagnostic (5a) then selection (5b). Issue #96.*

Neither table has explicit nulls. But the candidates table (issue #22) **codes** "not available" as a fill value: `rank_covis` and `rank_pop` are 201 when the candidate was not returned by that retriever. Those fills are structural missing values, and 5a/5b treat them as such. Cross-validation is 3-fold grouped by session on a sample of 6,000 queries (1.2 million rows); two model families at defaults (logistic regression with standardisation, LightGBM); metric PR-AUC (guide section 3, rule 14).

#### Why this step exists here
Missing values change every later comparison, so we work out why values are missing before choosing a treatment (5a), and then pick the treatment by cross-validated score, not by habit (5b). It comes after duplicates and before outliers and transforms because imputation must be fitted first for those steps to see complete columns.

#### Options considered
**5a Diagnose:** counts per column; co-missingness; relation of missingness to the target, to other columns and to time; classify each column as MCAR-, MAR- or MNAR-like.
**5b Treat (numeric columns):** drop columns above 30/50/80 percent missing; drop rows with any missing; mean; median; median + missing indicator; sentinel + indicator; segment median; KNN imputation; iterative (model-based) imputation; leave missing for the gradient boosting model. **Categorical:** most frequent level or an explicit `missing` level (not applicable: `category` has no missing values).

In [ ]:
import scipy.stats as st
from eda_cv import NUM, decide, evaluate_option, load_sample, make_folds, summarise

sample = load_sample(ROOT, 6000)
print(f"CV sample: {sample['session'].n_unique():,} queries, {sample.height:,} rows, {int(sample['y'].sum()):,} positives ({sample['y'].mean():.3%}); seed {SEED}")
full = pl.read_parquet(CAND, columns=["session", "label"] + NUM + ["category"])

# 5a. Diagnose
explicit_nulls = full.null_count().to_dicts()[0]
print("A. explicit nulls in the candidates table:", {k: v for k, v in explicit_nulls.items() if v}, "(none)" if not any(explicit_nulls.values()) else "")
coded = pl.DataFrame([
    ("rank_covis", "== 201 (not returned by co-visitation)", int((full["rank_covis"] == 201).sum())),
    ("rank_pop", "== 201 (not in the category popularity list)", int((full["rank_pop"] == 201).sum())),
    ("covis_max", "== 0 (no co-visitation with the prefix; a real zero)", int((full["covis_max"] == 0).sum())),
    ("hours_since_last_seen", "filled for unseen items (item_seen == 0)", int((full["item_seen"] == 0).sum())),
], schema=["column", "coded_missing_value", "rows"], orient="row").with_columns((pl.col("rows") / full.height).alias("share"))
print("A. coded missing values (share of all rows):"); print(coded)
save_table("step05_coded_missing", coded)
y_all = (full["label"] > 0)
rel = pl.DataFrame([{"column": c, "target_rate_missing": float(y_all.filter(m).mean()), "target_rate_present": float(y_all.filter(~m).mean()),
                     "mean_prefix_len_missing": float(full.filter(m)["prefix_len"].mean()), "mean_prefix_len_present": float(full.filter(~m)["prefix_len"].mean())}
                    for c, m in (("rank_covis", full["rank_covis"] == 201), ("rank_pop", full["rank_pop"] == 201))])
print("C. relation to the target and to the prefix length:"); print(rel)
save_table("step05_relation", rel)
both = full.select((pl.col("rank_covis") == 201).alias("covis_missing"), (pl.col("rank_pop") == 201).alias("pop_missing")).group_by("covis_missing", "pop_missing").len().with_columns((pl.col("len") / full.height).alias("share"))
print("B. co-missingness (rank_covis missing x rank_pop missing):"); print(both.sort("covis_missing", "pop_missing"))
print("phi correlation of the two missingness indicators:", round(float(np.corrcoef((full["rank_covis"] == 201).to_numpy(), (full["rank_pop"] == 201).to_numpy())[0, 1]), 3))

In [ ]:
# 5b. Options under the shared protocol. Missing = rank_covis (77%) and rank_pop (12%) set to NaN.
MISS = ["rank_covis", "rank_pop"]
mi = [NUM.index(c) for c in MISS]
sample_m = sample.with_columns([pl.when(pl.col(c) == 201).then(None).otherwise(pl.col(c)).alias(c) for c in MISS])
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer


def arrays(tr, va):
    return (tr.select(NUM).to_numpy().astype(np.float32), tr["y"].to_numpy(), va.select(NUM).to_numpy().astype(np.float32), va["y"].to_numpy())


def with_fill(fill_fn, indicator=False, drop_cols=()):
    """Generic prep: fill_fn(xtr, xva, tr, va) returns filled copies; optional indicator columns; optional dropped columns."""
    def prep(tr, va):
        xtr, ytr, xva, yva = arrays(tr, va)
        mtr, mva = np.isnan(xtr[:, mi]), np.isnan(xva[:, mi])
        xtr, xva = fill_fn(xtr, xva, tr, va)
        if indicator:
            xtr, xva = np.hstack([xtr, mtr.astype(np.float32)]), np.hstack([xva, mva.astype(np.float32)])
        if drop_cols:
            keep = [i for i in range(xtr.shape[1]) if i not in {NUM.index(c) for c in drop_cols}]
            xtr, xva = xtr[:, keep], xva[:, keep]
        return xtr, ytr, xva, yva
    return prep


def stat_fill(stat):
    def fn(xtr, xva, tr, va):
        xtr, xva = xtr.copy(), xva.copy()
        for i in mi:
            v = float(stat(xtr[~np.isnan(xtr[:, i]), i]))
            xtr[np.isnan(xtr[:, i]), i] = v; xva[np.isnan(xva[:, i]), i] = v
        return xtr, xva
    return fn


def const_fill(value):
    def fn(xtr, xva, tr, va):
        xtr, xva = xtr.copy(), xva.copy()
        for i in mi:
            xtr[np.isnan(xtr[:, i]), i] = value; xva[np.isnan(xva[:, i]), i] = value
        return xtr, xva
    return fn


def segment_median(xtr, xva, tr, va):
    xtr, xva = xtr.copy(), xva.copy()
    cat_tr, cat_va = tr["category"].to_numpy(), va["category"].to_numpy()
    for i in mi:
        glob = float(np.nanmedian(xtr[:, i]))
        med = {c: float(np.nanmedian(xtr[(cat_tr == c) & ~np.isnan(xtr[:, i]), i])) if ((cat_tr == c) & ~np.isnan(xtr[:, i])).any() else glob for c in np.unique(cat_tr)}
        for x, cat in ((xtr, cat_tr), (xva, cat_va)):
            m = np.isnan(x[:, i])
            x[m, i] = [med.get(c, glob) for c in cat[m]]
    return xtr, xva


def native(tr, va):                       # leave missing: only the gradient boosting model can use NaN
    xtr, ytr, xva, yva = arrays(tr, va)
    return {"gbm": (xtr, ytr, xva, yva)}


shares = {c: float(sample_m[c].null_count() / sample_m.height) for c in MISS}
dropped_30 = [c for c in MISS if shares[c] > 0.30]; dropped_50 = [c for c in MISS if shares[c] > 0.50]; dropped_80 = [c for c in MISS if shares[c] > 0.80]
print("missing shares in the sample:", {k: round(v, 3) for k, v in shares.items()}, "-> dropped at 30%:", dropped_30, " at 50%:", dropped_50, " at 80%:", dropped_80)
opts = {
    "leave missing (native, gbm only)": native,
    "sentinel 201 + indicator": with_fill(const_fill(201.0), indicator=True),
    "sentinel 201 (no indicator)": with_fill(const_fill(201.0)),
    "mean": with_fill(stat_fill(np.mean)),
    "median": with_fill(stat_fill(np.median)),
    "median + indicator": with_fill(stat_fill(np.median), indicator=True),
    "segment median (by category)": with_fill(segment_median),
    "drop columns > 30% missing (+ median fill of the rest)": with_fill(stat_fill(np.median), drop_cols=dropped_30),
    "drop columns > 50% missing (+ median fill of the rest)": with_fill(stat_fill(np.median), drop_cols=dropped_50),
    "keep columns up to 80% missing (median fill)": with_fill(stat_fill(np.median), drop_cols=dropped_80),
}
folds = make_folds(sample_m)
def run_all():
    rows = []
    for name, prep in opts.items():
        t0 = time.time()
        rows += evaluate_option(name, prep, sample_m, folds)
        print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv5 = cached("step05_cv_rows", run_all)
sum5 = summarise(cv5.to_dicts())
print(sum5.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc", "n_features").sort("family", "pr_auc", descending=[False, True]))

In [ ]:
# drop rows with any missing: changes the sample, so it is reported but not ranked
def drop_rows_option():
    rows = []
    def prep(tr, va):
        tr, va = tr.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS)), va.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS))
        return arrays(tr, va)
    rows += evaluate_option("drop rows with any missing (not ranked)", prep, sample_m, folds)
    return pl.DataFrame(rows)
dr = cached("step05_cv_droprows", drop_rows_option)
kept = sample_m.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS))
print(f"drop rows: keeps {kept.height / sample_m.height:.1%} of rows and {kept['y'].sum() / sample_m['y'].sum():.1%} of the positives")
print(summarise(dr.to_dicts()).select("option", "family", "pr_auc", "pr_auc_sd", "rows_train"))

In [ ]:
# KNN and iterative imputation are expensive: run on a 500-query sample, next to two cheap options on the SAME rows
small = load_sample(ROOT, 500).with_columns([pl.when(pl.col(c) == 201).then(None).otherwise(pl.col(c)).alias(c) for c in MISS])
folds_small = make_folds(small)
def knn_fill(xtr, xva, tr, va):
    ref = xtr[np.random.default_rng(SEED).choice(len(xtr), size=min(8000, len(xtr)), replace=False)]
    k = KNNImputer(n_neighbors=5).fit(ref)
    return k.transform(xtr), k.transform(xva)
def iter_fill(xtr, xva, tr, va):
    sub = xtr[np.random.default_rng(SEED).choice(len(xtr), size=min(20000, len(xtr)), replace=False)]
    imp = IterativeImputer(max_iter=5, random_state=SEED).fit(sub)
    return imp.transform(xtr), imp.transform(xva)
small_opts = {"small sample: median + indicator": with_fill(stat_fill(np.median), indicator=True),
              "small sample: sentinel 201 + indicator": with_fill(const_fill(201.0), indicator=True),
              "small sample: KNN (k=5, 8,000 reference rows)": with_fill(knn_fill),
              "small sample: iterative (BayesianRidge)": with_fill(iter_fill)}
def run_small():
    rows = []
    for name, prep in small_opts.items():
        t0 = time.time(); rows += evaluate_option(name, prep, small, folds_small); print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
small_cv = cached("step05_cv_small", run_small)
sum_small = summarise(small_cv.to_dicts())
print(f"small sample: {small.height:,} rows, {int(small['y'].sum())} positives")
print(sum_small.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc"))

In [ ]:
# distribution check: do the imputed values resemble the observed ones? (fold 0 fit, imputed values in missing rows)
tr0, va0 = sample_m[folds[0][0]], sample_m[folds[0][1]]
xtr0, _, xva0, _ = arrays(tr0, va0)
observed = {c: xva0[~np.isnan(xva0[:, NUM.index(c)]), NUM.index(c)] for c in MISS}
rows = []
for name, fn in (("mean", stat_fill(np.mean)), ("median", stat_fill(np.median)), ("sentinel 201", const_fill(201.0)), ("segment median", segment_median)):
    _, filled = fn(xtr0, xva0, tr0, va0)
    for c in MISS:
        i = NUM.index(c); miss = np.isnan(xva0[:, i])
        rows.append({"option": name, "column": c, "imputed_mean": float(filled[miss, i].mean()), "observed_mean": float(observed[c].mean()),
                     "imputed_sd": float(filled[miss, i].std()), "observed_sd": float(observed[c].std()),
                     "ks_vs_observed": float(st.ks_2samp(filled[miss, i][:50000], observed[c][:50000]).statistic)})
dist5 = pl.DataFrame(rows); save_table("step05_distribution_check", dist5); print(dist5)

In [ ]:
complexity5 = {"leave missing (native, gbm only)": 0, "sentinel 201 (no indicator)": 1, "sentinel 201 + indicator": 1,
               "drop columns > 30% missing (+ median fill of the rest)": 1, "drop columns > 50% missing (+ median fill of the rest)": 1,
               "keep columns up to 80% missing (median fill)": 2, "mean": 2, "median": 2, "median + indicator": 2, "segment median (by category)": 3}
save_table("step05_cv_summary", sum5)
decisions5 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum5, fam, complexity5)
    decisions5[fam] = (w, tied, info)
    print(f"first pass, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied within noise: {len(tied)} options; winner by simplicity: {w}")

# Guide rule 6: for a close call, repeat the cross-validation with different fold seeds. Every option tied within noise at the first pass, 2 extra seeds.
top = set(decisions5["linear"][1]) | set(decisions5["gbm"][1])      # every option tied with the best at the first pass, in either family
def rerun5():
    rows = []
    for seed in (43, 44):
        f = make_folds(sample_m, seed=seed)
        for name in sorted(top):
            for r in evaluate_option(name, opts[name], sample_m, f):
                rows.append({**r, "fold": r["fold"] + 3 * (seed - 42)})
    return pl.DataFrame(rows)
rep5 = cached("step05_cv_repeat", rerun5)
all5 = pl.concat([cv5.filter(pl.col("option").is_in(sorted(top))), rep5.select(cv5.columns)])
sum5r = summarise(all5.to_dicts()); save_table("step05_cv_summary_repeated", sum5r)
print("repeated CV (3 fold seeds x 3 folds = 9 folds) for the top options:"); print(sum5r.select("option", "family", "pr_auc", "pr_auc_sd", "folds").sort("family", "pr_auc", descending=[False, True]))
final5 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum5r, fam, complexity5)
    final5[fam] = (w, tied, info)
    print(f"after repeating, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {tied}; winner by simplicity: {w}")

In [ ]:
wl, tl, il = final5["linear"]; wg, tg, ig = final5["gbm"]
rc = rel.filter(pl.col("column") == "rank_covis").row(0, named=True); rp = rel.filter(pl.col("column") == "rank_pop").row(0, named=True)
knn_l = sum_small.filter((pl.col("option").str.contains("KNN")) & (pl.col("family") == "linear"))["pr_auc"][0]; med_l = sum_small.filter((pl.col("option").str.contains("median")) & (pl.col("family") == "linear"))["pr_auc"][0]
it_l = sum_small.filter((pl.col("option").str.contains("iterative")) & (pl.col("family") == "linear"))["pr_auc"][0]
e5 = log_step(5, "Missing values", "Diagnostic (5a) then Selection (5b)",
    "5a: A counts, B co-missingness, C relation to target/prefix length. 5b (10 options, both families, 3-fold grouped CV, 6,000 queries): leave missing (native GBM), sentinel 201 with and without indicator, mean, median, median + indicator, segment median, drop columns above 30% / 50% missing, keep up to 80% (median fill); drop rows (reported, not ranked); KNN (k=5) and iterative imputation (on a 500-query sample, next to two cheap options on the same rows). Close calls repeated with 2 extra fold seeds. Categorical fills: not applicable (category has no missing values).",
    f"No explicit nulls in either table. The candidates table CODES missing values as 201: rank_covis in {coded['share'][0]:.1%} of rows and rank_pop in {coded['share'][1]:.1%}. Both are informative: target rate {rc['target_rate_missing']:.2%} when rank_covis is missing vs {rc['target_rate_present']:.2%} present; {rp['target_rate_missing']:.2%} vs {rp['target_rate_present']:.2%} for rank_pop. The two are almost mutually exclusive (phi -0.685): a candidate comes from the co-visitation list, the popularity list, or both, so it is missing from at most one. Classification: MNAR-like (informative, structural). Linear model: median + indicator, sentinel + indicator and sentinel alone are tied at PR-AUC about 0.048; mean, median and dropping the column are lower (0.042-0.044). Gradient boosting: all options within noise (PR-AUC 0.034-0.039). KNN and iterative imputation do not beat median + indicator on the small sample (linear {knn_l:.3f} and {it_l:.3f} vs {med_l:.3f}). Dropping rows keeps only 11.1% of rows and 44.4% of positives (PR-AUC not comparable). No imputed distribution matches the observed one (KS 0.40-1.0), which is expected when the value means 'not in the list'.",
    f"Linear model: {wl}. Gradient boosting: {wg}. One variant per model family in the preprocessing spec.",
    f"Guide rules: options within the larger fold standard deviation of the best are tied (linear: {len(tl)} options, gbm: {len(tg)} options after repeating the CV with 3 fold seeds); among tied options take the simplest (leave as is < fixed rule < learned statistic < model). Missingness is informative, so an option that keeps the information (an indicator, or a sentinel the trees can split on) is preferred.",
    "Drop columns above 30% or 50%: rejected, it removes the strongest missingness signal (rank_covis is missing for 77%), PR-AUC 0.042 (linear) and 0.034 (gbm). Keeping columns up to 80%: identical to the median fill (no column is above 80%). Mean/median/segment median: lower for the linear model because they erase the fact that the candidate was not returned. KNN and iterative: no better than the simple options and far more expensive (KNN took 86 s on 100,000 rows). Drop rows: rejected, it discards 89% of rows and 56% of positives and would bias the sample.",
    "Linear: Medium (three tied options, chosen by the simplicity rule). Gradient boosting: Low (all options tied: the choice does not matter much, the tie-break picked the simplest). The gbm choice goes on the needs-human-review list.",
    "Step 6-7 use the coded values as they are; Step 14 keeps in mind that most negatives have rank_covis = 201, so one code carries much of the signal. The two coded columns must be treated the same in training and serving (issue #58).")
print(e5)

#### Chosen approach and rationale
**Decision.** Linear model: sentinel 201 + indicator. Gradient boosting: leave missing (native, gbm only). One variant per model family in the preprocessing spec.

**Why.** Guide rules: options within the larger fold standard deviation of the best are tied (linear: 3 options, gbm: 8 options after repeating the CV with 3 fold seeds); among tied options take the simplest (leave as is < fixed rule < learned statistic < model). Missingness is informative, so an option that keeps the information (an indicator, or a sentinel the trees can split on) is preferred.

**Rejected.** Drop columns above 30% or 50%: rejected, it removes the strongest missingness signal (rank_covis is missing for 77%), PR-AUC 0.042 (linear) and 0.034 (gbm). Keeping columns up to 80%: identical to the median fill (no column is above 80%). Mean/median/segment median: lower for the linear model because they erase the fact that the candidate was not returned. KNN and iterative: no better than the simple options and far more expensive (KNN took 86 s on 100,000 rows). Drop rows: rejected, it discards 89% of rows and 56% of positives and would bias the sample.

**Confidence.** Linear: Medium (three tied options, chosen by the simplicity rule). Gradient boosting: Low (all options tied: the choice does not matter much, the tie-break picked the simplest). The gbm choice goes on the needs-human-review list.

#### Interpretation: what we understand from the results
The tables have no explicit nulls, but the candidate table hides a large amount of structural missingness: `rank_covis` is missing (coded 201) for 77% of the rows and `rank_pop` for 12%. This is not random. A candidate comes from the co-visitation list, the popularity list, or both, so it is missing from at most one, and the target rate is 11 times lower when the co-visitation rank is missing (0.13% vs 1.48%). That makes the missingness itself the signal, which is why every option that keeps it (a sentinel or an indicator) beats the ones that erase it for the linear model, and why dropping the column or the rows is the wrong answer. For the gradient boosting model nothing matters much (all options are within noise), because trees can split on the sentinel by themselves. The expensive model-based imputers (KNN, iterative) did not beat the simple options.

#### Impact on next steps
Step 6-7 use the coded values as they are; Step 14 keeps in mind that most negatives have rank_covis = 201, so one code carries much of the signal. The two coded columns must be treated the same in training and serving (issue #58).

#### Out of scope
- The two coded columns must be produced the same way at serving time (issue #58, feature store).
- Whether the pool composition (why a candidate is missing from one list) should itself be a feature is a Step 9 question.

## Step 6. Univariate analysis
*Kind: diagnostic. Issue #97.* Every numeric feature of the candidates table (18) and the categorical column, plus the shape of the events table (session length, item frequency, events per day).

#### Why this step exists here
After cleaning and missing values, we look at every variable on its own so that we know its shape before comparing it with anything else. The shapes decide which transformations to test in Step 8, which columns to flag as zero-inflated and which to drop.

#### Options considered
**Numeric columns:** A summary statistics (mean, sd, percentiles, skew, kurtosis, share of zeros); B plots (histogram, box plot, ECDF); C normality checks (Q-Q plot, Shapiro test on a sample).
**Categorical columns:** A level counts including missing; B cardinality and rare levels; C consistency (spellings, case).
**Also:** the shape of the events table (events per session, per item and per day).

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, shapiro

qs = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
rows = []
for c in NUM:
    s = full[c].cast(pl.Float64)
    r = {"column": c, "mean": s.mean(), "sd": s.std(), "min": s.min()}
    r.update({f"p{int(q * 100)}": s.quantile(q) for q in qs})
    r.update({"max": s.max(), "skew": s.skew(), "kurtosis": kurtosis(s.to_numpy()[:2_000_000]), "share_zero": float((s == 0).mean()), "n_unique": s.n_unique(),
              "top_value_share": float(s.value_counts().sort("count", descending=True)["count"][0] / full.height)})
    rows.append(r)
summ6 = pl.DataFrame(rows); save_table("step06_numeric_summary", summ6)
with pl.Config(tbl_cols=20, tbl_width_chars=260, fmt_float="mixed"):
    print("A. summary statistics of the 18 numeric features (8,000,000 rows):"); print(summ6.select("column", "mean", "sd", "min", "p50", "p99", "max", "skew", "share_zero", "n_unique", "top_value_share"))

In [ ]:
# B. plots on a 300,000-row sample: histogram, box plot and empirical CDF for every numeric feature
samp = full.sample(300_000, seed=SEED)
fig, axes = plt.subplots(6, 3, figsize=(13, 17))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    v = samp[c].cast(pl.Float64).to_numpy()
    ax.hist(v, bins=60); ax.set_yscale("log"); ax.set_title(f"{c} (skew {summ6.filter(pl.col('column') == c)['skew'][0]:.1f})", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_histograms.png", dpi=80); plt.close()
fig, axes = plt.subplots(6, 3, figsize=(13, 17))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    v = np.sort(samp[c].cast(pl.Float64).to_numpy()[:100_000]); ax.plot(v, np.linspace(0, 1, len(v))); ax.set_title(f"ECDF {c}", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_ecdf.png", dpi=80); plt.close()
fig, axes = plt.subplots(3, 6, figsize=(17, 8))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    ax.boxplot(samp[c].cast(pl.Float64).to_numpy(), vert=True); ax.set_title(c, fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_boxplots.png", dpi=80); plt.close()
print("saved eda/figures/step06_histograms.png, step06_ecdf.png, step06_boxplots.png")

# C. normality: Q-Q plots of the 4 most skewed features and a Shapiro test on 5,000 rows for all (weigh the plot, not the p-value)
top_skew = summ6.sort(summ6["skew"].abs(), descending=True)["column"].to_list()[:4]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, c in zip(axes, top_skew, strict=True):
    st.probplot(samp[c].cast(pl.Float64).to_numpy()[:20_000], dist="norm", plot=ax); ax.set_title(f"Q-Q {c}", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_qq.png", dpi=80); plt.close()
sh = pl.DataFrame([{"column": c, "shapiro_p": float(shapiro(samp[c].cast(pl.Float64).to_numpy()[:5000]).pvalue)} for c in NUM]); save_table("step06_shapiro", sh)
print("C. Shapiro p-values on 5,000 rows (all far below 0.05: none is close to normal):", int((sh["shapiro_p"] < 0.05).sum()), "of", sh.height, "reject normality")

In [ ]:
# categorical columns: category (80 levels), event type, and the item id as a cardinality reference
lv = full.group_by("category").len().sort("len", descending=True).with_columns((pl.col("len") / full.height).alias("share"))
cat6 = pl.DataFrame([
    {"column": "category", "n_levels": lv.height, "missing": int(full["category"].null_count()), "top_level_share": float(lv["share"][0]), "levels_under_1pct": int((lv["share"] < 0.01).sum()),
     "smallest_level_share": float(lv["share"][-1])},
    {"column": "type (events)", "n_levels": 3, "missing": 0, "top_level_share": 0.898, "levels_under_1pct": 0, "smallest_level_share": 0.0239},
    {"column": "aid (item id, events)", "n_levels": 1070425, "missing": 0, "top_level_share": float(1 / 1070425 * 0 + ev.group_by("aid").len().sort("len", descending=True)["len"][0] / ev.height), "levels_under_1pct": 1070425 - 0, "smallest_level_share": 1 / ev.height},
])
save_table("step06_categorical", cat6); print("categorical columns:"); print(cat6)
print("C. consistency: category is a synthetic integer id (no spellings to merge); event type is a fixed 3-code column. Nothing to merge.")
# the events table
lengths = ev.group_by("session").len()["len"]
item_freq = ev.group_by("aid").len()["len"]
per_day = ev.with_columns(((pl.col("ts") - START_MS) // DAY_MS).alias("day")).group_by("day").len().sort("day")
events6 = pl.DataFrame([
    {"quantity": "events per session", "min": lengths.min(), "median": lengths.median(), "mean": lengths.mean(), "p99": lengths.quantile(0.99), "max": lengths.max(), "skew": lengths.skew()},
    {"quantity": "events per item", "min": item_freq.min(), "median": item_freq.median(), "mean": item_freq.mean(), "p99": item_freq.quantile(0.99), "max": item_freq.max(), "skew": item_freq.skew()},
    {"quantity": "events per day", "min": per_day["len"].min(), "median": per_day["len"].median(), "mean": per_day["len"].mean(), "p99": per_day["len"].quantile(0.99), "max": per_day["len"].max(), "skew": per_day["len"].skew()},
])
save_table("step06_events", events6); print("events table shape:"); print(events6)
print("events per day:", per_day["len"].to_list())

In [ ]:
# actions
skewed = summ6.filter(pl.col("skew").abs() > 1)["column"].to_list()
zero_spike = summ6.filter(pl.col("share_zero") > 0.5)["column"].to_list()
constant = summ6.filter(pl.col("n_unique") == 1)["column"].to_list()
near_const = summ6.filter((pl.col("top_value_share") > 0.999) & (pl.col("n_unique") > 1))["column"].to_list()
print("skew > 1 (candidates for Step 8):", skewed)
print("more than half zeros (candidate 'is zero' flag):", zero_spike)
print("constant columns (to drop):", constant, "| near-constant (> 99.9% one value):", near_const)
nc = []
for c in near_const:
    top = full[c].value_counts().sort("count", descending=True)["".join(["" if False else c])][0]
    m = full[c] == top
    nc.append({"column": c, "target_rate_common": float((full["label"].filter(m) > 0).mean()), "target_rate_rare": float((full["label"].filter(~m) > 0).mean()), "rare_rows": int((~m).sum())})
if nc:
    print("near-constant columns: target rate of the common vs the rare value:"); print(pl.DataFrame(nc))
actions6 = pl.DataFrame({"action": ["mark for the Step 8 transform test", "candidate 'is zero' flag", "drop (constant)", "check rare value vs target (near-constant)", "ID-like, exclude from features"],
                         "columns": [", ".join(skewed), ", ".join(zero_spike), ", ".join(constant) or "-", ", ".join(near_const) or "-", "session, aid"]})
save_table("step06_actions", actions6); print(actions6)

In [ ]:
e6 = log_step(6, "Univariate analysis", "Diagnostic",
    "Numeric: A summary statistics (mean, sd, min, percentiles 1-99, max, skew, kurtosis, share of zeros), B histogram + box plot + ECDF for all 18 features (eda/figures/step06_*.png), C Q-Q plots of the 4 most skewed features and a Shapiro test on 5,000 rows. Categorical: A level counts, B cardinality and rare levels, C consistency. Also the shape of the events table (events per session, per item, per day).",
    f"{len(skewed)} of 18 numeric features have |skew| > 1 ({', '.join(skewed)}). {len(zero_spike)} have more than half zeros ({', '.join(zero_spike)}). Constant: {', '.join(constant) or 'none'}; near-constant (> 99.9% one value): {', '.join(near_const) or 'none'}. Every Shapiro test rejects normality (p < 0.05) on 5,000 rows, as expected for counts and ranks; the Q-Q plots show heavy right tails for the most skewed features. `category` has {lv.height} levels, none missing, largest level {lv['share'][0]:.1%}. Events: median {lengths.median():.0f} events per session (max {lengths.max()}), median {item_freq.median():.0f} events per item (max {item_freq.max():,}).",
    "Actions: mark the skewed features for the transform test in Step 8; add an 'is zero' flag for the zero-spike columns if Step 9 shows the zero group differs; drop the constant column(s); check the target rate of near-constant columns before dropping; keep ids out of the features.",
    "The guide's rules for the actions above were applied as written; the Shapiro p-values were read with the plots, not on their own (with millions of rows every test rejects).",
    "Nothing to reject: every option was run and describes a different aspect. Dropping near-constant columns straight away was rejected because a rare value can be the signal (checked against the target above).",
    "High", "Step 8 tests transforms on the skewed features; Step 7 uses the tails seen here; Step 13 handles category (80 levels) and aid.")
print(e6)

#### Chosen approach and rationale
**Decision.** Actions: mark the skewed features for the transform test in Step 8; add an 'is zero' flag for the zero-spike columns if Step 9 shows the zero group differs; drop the constant column(s); check the target rate of near-constant columns before dropping; keep ids out of the features.

**Why.** The guide's rules for the actions above were applied as written; the Shapiro p-values were read with the plots, not on their own (with millions of rows every test rejects).

**Rejected.** Nothing to reject: every option was run and describes a different aspect. Dropping near-constant columns straight away was rejected because a rare value can be the signal (checked against the target above).

**Confidence.** High

#### Interpretation: what we understand from the results
Most of the features are far from normal: 13 of 18 have |skew| above 1, and every Shapiro test rejects normality (with millions of rows that is expected, so the plots matter more). The features come in three kinds. Rank-like columns (`rank_covis`, `rank_pop`) are bounded 1-201 and piled up at 201 (the sentinel from Step 5). Count-like columns (`prefix_len`, `prefix_n_cart`, `prefix_n_order`) and co-visitation weights are very right-skewed and mostly zero (77% zeros for the co-visitation columns, which means 'no co-visited neighbour'). Rate-like columns (`cart_rate`, `order_rate`, `pop_cat_pct`) are skewed but bounded. One column, `item_seen`, is constant (always 1, because every candidate has been seen) and is dropped. `category` has 80 levels with no dominant one (largest 5.6%), and 34 levels are under 1%. On the events table, sessions are short (median 5 events) with a long tail up to 485, items are very uneven (median 3 events, maximum 8,071), and day 13 has 40% more events than an average day.

#### Impact on next steps
Step 8 tests transforms on the skewed features; Step 7 uses the tails seen here; Step 13 handles category (80 levels) and aid.

#### Out of scope
- Which transform helps which model family is decided in Step 8, not here.
- Whether a zero-inflated column needs an `is zero` flag is decided from the target relationship in Step 9.

## Step 7. Outliers
*Kind: diagnostic (7a) then selection (7b). Issue #98.* Candidate features, development rows only.

#### Why this step exists here
Extreme values can be errors, which we must correct, or real and informative, which we must keep. We separate the two before transforming (Step 8) because a cap or a transform changes what the model sees. It comes after the univariate step, which showed us the tails.

#### Options considered
**7a Detect:** IQR fences; z-score above 3; modified z-score above 3.5 (skipped when the median absolute deviation is zero); 1st/99th percentile cut; Isolation Forest (multivariate); Local Outlier Factor (multivariate).
**7b Treat:** keep as is; cap at the 1st/99th percentile; cap at the 5th/95th; log or power transform; remove flagged rows (training folds only); keep and add an outlier flag feature; model the extreme segment separately (only if flagged rows form a coherent population).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor


# 7a. Detect: six methods. Univariate methods on all 8,000,000 rows, multivariate ones on samples.
def univariate_flags(s):
    x = s.cast(pl.Float64)
    q1, q3, med = x.quantile(0.25), x.quantile(0.75), x.median()
    iqr = q3 - q1
    mad = (x - med).abs().median()
    return {"iqr": (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr), "zscore": ((x - x.mean()) / x.std()).abs().fill_nan(0) > 3,      # a constant column has sd 0: no flags
            "modified_z": (0.6745 * (x - med).abs() / mad) > 3.5 if mad and mad > 0 else pl.Series([False] * len(x)),
            "percentile": (x < x.quantile(0.01)) | (x > x.quantile(0.99))}, mad
rows7, flagsets = [], {}
for c in NUM:
    flags, mad = univariate_flags(full[c])
    row = {"column": c, "mad_zero": bool(mad == 0)}
    row.update({m: float(f.mean()) for m, f in flags.items()})
    a, b_ = flags["iqr"].to_numpy(), flags["zscore"].to_numpy()
    row["jaccard_iqr_z"] = float((a & b_).sum() / max((a | b_).sum(), 1))
    rows7.append(row)
flag7 = pl.DataFrame(rows7); save_table("step07_flag_shares", flag7)
with pl.Config(tbl_cols=20, tbl_width_chars=220, fmt_float="mixed"):
    print("A-D. share of rows flagged per method and column:"); print(flag7)
print("modified z-score skipped (MAD = 0) for:", flag7.filter(pl.col("mad_zero"))["column"].to_list())

In [ ]:
# E. Isolation Forest (200,000-row sample) and F. Local Outlier Factor (20,000-row sample)
X = full.select(NUM).to_numpy().astype(np.float32)
rs = np.random.default_rng(SEED)
i_if = rs.choice(len(X), 200_000, replace=False); i_lof = i_if[:20_000]
iso = IsolationForest(n_estimators=100, random_state=SEED, n_jobs=4).fit(X[i_if])
f_if = iso.predict(X[i_if]) == -1
Xs = (X[i_lof] - X[i_lof].mean(0)) / (X[i_lof].std(0) + 1e-9)
f_lof = LocalOutlierFactor(n_neighbors=20).fit_predict(Xs) == -1
f_if_on_lof = f_if[:20_000]
overlap = float((f_if_on_lof & f_lof).sum() / max((f_if_on_lof | f_lof).sum(), 1))
y_if = (full["label"].to_numpy()[i_if] > 0)
multi7 = pl.DataFrame([
    {"method": "Isolation Forest (200,000 rows)", "flagged": int(f_if.sum()), "share": float(f_if.mean()), "target_rate_flagged": float(y_if[f_if].mean()), "target_rate_rest": float(y_if[~f_if].mean())},
    {"method": "Local Outlier Factor (20,000 rows)", "flagged": int(f_lof.sum()), "share": float(f_lof.mean()), "target_rate_flagged": float((full["label"].to_numpy()[i_lof] > 0)[f_lof].mean()), "target_rate_rest": float((full["label"].to_numpy()[i_lof] > 0)[~f_lof].mean())},
])
save_table("step07_multivariate", multi7); print(multi7); print(f"overlap (Jaccard) of Isolation Forest and LOF flags on the same 20,000 rows: {overlap:.2f}")

# target rate of rows flagged by the IQR rule in any column, and how much they matter
iqr_any = np.zeros(len(X), dtype=bool)
for j, c in enumerate(NUM):
    x = X[:, j].astype(np.float64); q1, q3 = np.quantile(x[::40], [0.25, 0.75]); iqr_any |= (x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))
yy = full["label"].to_numpy() > 0
print(f"rows flagged by the IQR rule in at least one column: {iqr_any.mean():.1%}; target rate flagged {yy[iqr_any].mean():.3%} vs rest {yy[~iqr_any].mean():.3%}")

In [ ]:
# inspect flagged rows: are they impossible values (errors), real extremes or unknown?
domain = {"cart_rate": (0, np.inf), "order_rate": (0, 1), "pop_cat_pct": (0, 1), "prefix_share_same_cat": (0, 1), "hours_since_last_seen": (0, np.inf), "prefix_len": (1, np.inf), "rank_covis": (1, 201), "rank_pop": (1, 201)}
imp = [{"column": c, "outside_valid_range": int(((full[c] < lo) | (full[c] > hi)).sum())} for c, (lo, hi) in domain.items()]
imp7 = pl.DataFrame(imp); save_table("step07_impossible", imp7); print("impossible values (outside the valid domain):"); print(imp7)
top_cols = flag7.sort("iqr", descending=True)["column"].to_list()[:3]
for c in top_cols:
    v = full[c].to_numpy(); q1, q3 = np.quantile(v[::40], [0.25, 0.75]); hi = q3 + 1.5 * (q3 - q1)
    print(f"{c}: flagged rows have values {np.round(np.sort(v[v > hi])[:3], 2)} ... {np.round(np.sort(v[v > hi])[-3:], 2)} (upper fence {hi:.2f}); positives among flagged {yy[v > hi].mean():.3%} vs overall {yy.mean():.3%}")
labels7 = pl.DataFrame({"pattern": ["long sessions (prefix_len, prefix_n_cart, prefix_n_order in the far tail)", "very large co-visitation weights (covis_*)", "cart_rate above 1 (carts per click; 1,300 rows, max 2.78) and high order_rate", "rank/popularity extremes"],
                        "label": ["real extreme", "real extreme", "real extreme", "real extreme"],
                        "evidence": ["session lengths up to 495 events exist in the events table (Step 6)", "weights are sums over up to 30 events and 10 offsets, so large values are possible", "cart_rate is carts per click, so it can exceed 1 when an item is carted more than clicked (Step 2D: carts without clicks exist); order_rate stays in [0, 1]", "ranks are bounded 1-201 by construction"]})
save_table("step07_pattern_labels", labels7); print(labels7)

In [ ]:
# 7b. Treat, compared on the shared protocol (both families). Cap thresholds are fitted inside each fold.
def caps(q_lo, q_hi):
    def prep(tr, va):
        xtr, ytr, xva, yva = arrays(tr, va)
        lo, hi = np.quantile(xtr, q_lo, axis=0), np.quantile(xtr, q_hi, axis=0)
        return np.clip(xtr, lo, hi), ytr, np.clip(xva, lo, hi), yva
    return prep
def keep(tr, va): return arrays(tr, va)
def logt(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    idx = [NUM.index(c) for c in skewed if xtr[:, NUM.index(c)].min() >= 0]
    xtr, xva = xtr.copy(), xva.copy(); xtr[:, idx] = np.log1p(xtr[:, idx]); xva[:, idx] = np.log1p(xva[:, idx])
    return xtr, ytr, xva, yva
def flag_feature(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    lo, hi = np.quantile(xtr, 0.01, axis=0), np.quantile(xtr, 0.99, axis=0)
    ftr, fva = ((xtr < lo) | (xtr > hi)).sum(1, keepdims=True), ((xva < lo) | (xva > hi)).sum(1, keepdims=True)
    return np.hstack([xtr, ftr]), ytr, np.hstack([xva, fva]), yva
def remove_flagged(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    lo, hi = np.quantile(xtr, 0.01, axis=0), np.quantile(xtr, 0.99, axis=0)
    bad = ((xtr < lo) | (xtr > hi)).sum(1) >= 3
    return xtr[~bad], ytr[~bad], xva, yva
xs_all = sample.select(NUM).to_numpy().astype(np.float32)
lo_, hi_ = np.quantile(xs_all, 0.01, axis=0), np.quantile(xs_all, 0.99, axis=0)
share_removed = float((((xs_all < lo_) | (xs_all > hi_)).sum(1) >= 3).mean())
print(f"rows removed by 'flagged in 3+ columns (outside the 1st/99th percentile)': {share_removed:.1%} of rows (limit 5%)")
opts7 = {"keep as is": keep, "cap at 1st/99th percentile": caps(0.01, 0.99), "cap at 5th/95th percentile": caps(0.05, 0.95), "log1p of the skewed features": logt,
         "keep + outlier-count flag feature": flag_feature}
if share_removed <= 0.05:
    opts7["remove flagged rows (training folds only)"] = remove_flagged
def run7():
    rows = []
    for name, prep in opts7.items():
        t0 = time.time(); rows += evaluate_option(name, prep, sample, make_folds(sample)); print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv7 = cached("step07_cv_rows", run7)
sum7 = summarise(cv7.to_dicts()); save_table("step07_cv_summary", sum7)
print(sum7.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc", "rows_train").sort("family", "pr_auc", descending=[False, True]))
complexity7 = {"keep as is": 0, "cap at 1st/99th percentile": 2, "cap at 5th/95th percentile": 2, "log1p of the skewed features": 1, "keep + outlier-count flag feature": 2, "remove flagged rows (training folds only)": 3}
decisions7 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum7, fam, complexity7, exclude=("remove flagged rows (training folds only)",))
    decisions7[fam] = (w, tied, info); print(f"{fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {len(tied)}; winner: {w}")

In [ ]:
wl7, tl7, il7 = decisions7["linear"]; wg7, tg7, ig7 = decisions7["gbm"]
def sc(opt, fam): return float(sum7.filter((pl.col("option") == opt) & (pl.col("family") == fam))["pr_auc"][0])
mv = multi7.row(0, named=True)
e7 = log_step(7, "Outliers", "Diagnostic (7a) then Selection (7b)",
    "7a: IQR fences, z-score > 3, modified z-score > 3.5 (skipped where MAD = 0), 1st/99th percentile cut, Isolation Forest (200,000 rows), Local Outlier Factor (20,000 rows); impossible-value checks; flagged-row patterns; target rate of flagged rows. 7b (3-fold grouped CV, both families): keep as is, cap 1/99, cap 5/95, log1p of the skewed features, keep + outlier-count flag, remove flagged rows (in 3+ columns, training folds only, 1.8% of rows). 'Model the extreme segment separately' was considered and not run (reason below).",
    f"Flag shares differ a lot between methods (IQR 0-23%, z-score 0-2.8%, percentile 1%); the IQR share for the co-visitation columns (23%) is an artefact of zero inflation (77% zeros, so every non-zero value is beyond the fence). The modified z-score is undefined (MAD = 0) for 7 columns (rank_covis, item_seen, the three covis columns, prefix_n_cart, prefix_n_order). Multivariate: Isolation Forest flags {mv['share']:.1%} of rows and LOF 1.4%; they agree very little (Jaccard 0.05). Impossible values: none. cart_rate above 1 in 1,300 rows (max 2.78) is not an error: it is carts per click and exceeds 1 when an item is carted more than clicked. Every flagged pattern is a real extreme (long sessions, large co-visitation weights, high cart/order rates). Flagged rows carry the signal: target rate {mv['target_rate_flagged']:.2%} for Isolation Forest flags vs {mv['target_rate_rest']:.2%} for the rest, and 0.88% vs 0.15% for rows outside the IQR fence in any column. 7b: linear PR-AUC log1p {sc('log1p of the skewed features', 'linear'):.4f}, keep {sc('keep as is', 'linear'):.4f}, flag feature {sc('keep + outlier-count flag feature', 'linear'):.4f}, cap 1/99 {sc('cap at 1st/99th percentile', 'linear'):.4f}, cap 5/95 {sc('cap at 5th/95th percentile', 'linear'):.4f}, remove flagged {sc('remove flagged rows (training folds only)', 'linear'):.4f}. Gradient boosting: keep {sc('keep as is', 'gbm'):.4f}, log1p {sc('log1p of the skewed features', 'gbm'):.4f}, flag {sc('keep + outlier-count flag feature', 'gbm'):.4f}, cap 1/99 {sc('cap at 1st/99th percentile', 'gbm'):.4f}, remove {sc('remove flagged rows (training folds only)', 'gbm'):.4f}, cap 5/95 {sc('cap at 5th/95th percentile', 'gbm'):.4f}.",
    f"Keep every row; no capping and no removal. Linear model: {wl7} (formalised in Step 8). Gradient boosting: {wg7}. No outlier flag feature.",
    f"Guide rules: errors become missing (none found); real extremes are kept for tree models, and for the linear model the choice among keep, cap and transform is by score: log1p beats keeping by {sc('log1p of the skewed features', 'linear') - sc('keep as is', 'linear'):.4f}, above the noise of {il7['noise']:.4f}. For the gradient boosting model {len(tg7)} options are tied and the simplest, keep as is, wins. Never remove rows because of their target or more than 5% of rows.",
    "Cap at 1st/99th and cap at 5th/95th percentile: rejected, lower PR-AUC for both families (the extreme values ARE the signal, so clipping them loses information). Remove flagged rows: rejected, lower for both families and it discards positives. Keep + flag feature: tied with keeping as is, so not added (simplicity). 'Model the extreme segment separately': not run: the flagged population is not coherent (the methods agree at Jaccard 0.05, IQR flags 40% of rows), and the guide only asks for it when flagged rows form a coherent population.",
    "High",
    "Step 8 formally tests transforms (log1p is the linear favourite here); Step 9 must not read the outlier rows as noise: they are where the positives are. Informational note for the human (goes on the review list): outlier rows have a 2x-7x higher target rate.")
print(e7)

#### Chosen approach and rationale
**Decision.** Keep every row; no capping and no removal. Linear model: log1p of the skewed features (formalised in Step 8). Gradient boosting: keep as is. No outlier flag feature.

**Why.** Guide rules: errors become missing (none found); real extremes are kept for tree models, and for the linear model the choice among keep, cap and transform is by score: log1p beats keeping by 0.0107, above the noise of 0.0029. For the gradient boosting model 4 options are tied and the simplest, keep as is, wins. Never remove rows because of their target or more than 5% of rows.

**Rejected.** Cap at 1st/99th and cap at 5th/95th percentile: rejected, lower PR-AUC for both families (the extreme values ARE the signal, so clipping them loses information). Remove flagged rows: rejected, lower for both families and it discards positives. Keep + flag feature: tied with keeping as is, so not added (simplicity). 'Model the extreme segment separately': not run: the flagged population is not coherent (the methods agree at Jaccard 0.05, IQR flags 40% of rows), and the guide only asks for it when flagged rows form a coherent population.

**Confidence.** High

#### Interpretation: what we understand from the results
The tails are not errors: no value is impossible, and every flagged pattern (very long sessions, large co-visitation weights, high cart and order rates) is a real extreme. The most important finding is that **the extremes are where the signal is**: rows outside the IQR fence in any column have a target rate of 0.88% against 0.15% for the rest (6 times), and Isolation Forest flags have 1.99% against 0.27%. So removing or clipping them throws away the positives, and the comparison confirms it: capping and removing lowered PR-AUC for both model families. The detectors disagree a lot (Jaccard 0.05 between the two multivariate methods; the IQR flags 23% of the co-visitation columns only because they are 77% zeros), which shows that 'outlier' depends heavily on the definition. A log transform, however, clearly helps the linear model (PR-AUC 0.059 against 0.048 for keeping the values), because it compresses the heavy tails without removing them. `cart_rate` above 1 turned out to be legitimate (carts per click can exceed 1). One assumption of mine was wrong here: I first treated values above 1 as impossible.

#### Impact on next steps
Step 8 formally tests transforms (log1p is the linear favourite here); Step 9 must not read the outlier rows as noise: they are where the positives are. Informational note for the human (goes on the review list): outlier rows have a 2x-7x higher target rate.

#### Out of scope
- The formal transform and scaler comparison is Step 8 (issue #99).
- Whether an outlier flag would help a different model is not tested beyond the tie found here.